In [2]:
from google.colab import drive 

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path

currDir = Path.cwd()
print(currDir)

/content


In [4]:
!ls 

drive  sample_data


In [5]:
!unzip /content/drive/MyDrive/finalDataset.zip -d /content/dataset

Archive:  /content/drive/MyDrive/finalDataset.zip
   creating: /content/dataset/finalDataset/
   creating: /content/dataset/finalDataset/2DMask/
  inflating: /content/dataset/finalDataset/2DMask/000000.mp4  
  inflating: /content/dataset/finalDataset/2DMask/000001.mp4  
  inflating: /content/dataset/finalDataset/2DMask/000002.mp4  
  inflating: /content/dataset/finalDataset/2DMask/000003.mp4  
  inflating: /content/dataset/finalDataset/2DMask/000004.mp4  
  inflating: /content/dataset/finalDataset/2DMask/000005.mp4  
  inflating: /content/dataset/finalDataset/2DMask/000006.mp4  
  inflating: /content/dataset/finalDataset/2DMask/000007.mp4  
  inflating: /content/dataset/finalDataset/2DMask/000008.mp4  
  inflating: /content/dataset/finalDataset/2DMask/000009.mp4  
  inflating: /content/dataset/finalDataset/2DMask/000010.mp4  
  inflating: /content/dataset/finalDataset/2DMask/000011.mp4  
  inflating: /content/dataset/finalDataset/2DMask/000012.mp4  
  inflating: /content/dataset/finalD

In [6]:
import cv2 as cv 
import os 
from pathlib import Path 
import shutil

splitRatio = 0.8

fileName = Path("/content/dataset/finalDataset")
numbers = []
for file in os.listdir(fileName):
    print(file)
    videoDis = fileName / file 
    os.makedirs(videoDis/"train", exist_ok= True)
    os.makedirs(videoDis/"val", exist_ok= True)
    files = sorted(os.listdir(videoDis))
    files = files[:-2]
    split = int(len(files) * splitRatio)
    print(split, len(files))
    train_vid = files[:split]
    val_vid = files[split:]
    print(len(train_vid), len(val_vid))
    print(videoDis)
    for vid in train_vid:
        shutil.move(videoDis/vid, videoDis/"train"/vid)
    for vid in val_vid:
        shutil.move(videoDis/vid, videoDis/"val"/vid)

2DMask
58 73
58 15
/content/dataset/finalDataset/2DMask
Real
48 61
48 13
/content/dataset/finalDataset/Real
Print
34 43
34 9
/content/dataset/finalDataset/Print
3DMask
43 54
43 11
/content/dataset/finalDataset/3DMask
Replay
46 58
46 12
/content/dataset/finalDataset/Replay


In [7]:
!git clone https://github.com/facenox/face-antispoof-onnx.git

Cloning into 'face-antispoof-onnx'...
remote: Enumerating objects: 361, done.
remote: Counting objects: 100% (201/201), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 361 (delta 69), reused 145 (delta 43), pack-reused 160 (from 3)
Receiving objects: 100% (361/361), 356.84 MiB | 7.48 MiB/s, done.
Resolving deltas: 100% (116/116), done.
Updating files: 100% (177/177), done.


In [8]:
%cd face-antispoof-onnx

/content/face-antispoof-onnx


In [9]:
!pip install -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 96.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 88.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.4 MB/s eta 0:00:00
  Created wheel for GPUtil: filename=GPUtil-1.4.0-py3-none-any.whl size=7392 sha256=ff00c8bf0b0869da289157d602723fe639648f0d9f08277a3249f7338c899d75
  Stored in directory: /root/.cache/pip/wheels/92/a8/b7/d8a067c31a74de9ca252bbe53dea5f896faabd25d55f541037
Successfully built GPUtil


In [10]:
%cd ..

/content


In [11]:
threshold = 0.5

In [12]:
import numpy as np
import sys

sys.path.append("/content/face-antispoof-onnx")
from pathlib import Path



from src.inference import (
    load_model,
)
from src.detection import load_detector

MODELS_DIR = currDir/"face-antispoof-onnx"/ "models"
DETECTOR_MODEL = MODELS_DIR / "detector_quantized.onnx"
LIVENESS_MODEL = MODELS_DIR / "best_model_quantized.onnx"



p = max(1e-6, min(1 - 1e-6, threshold))
logit_threshold = np.log(p / (1 - p))

face_detector = load_detector(DETECTOR_MODEL, (320, 320))
liveness_session, input_name = load_model(LIVENESS_MODEL)

if liveness_session is None or face_detector is None:
    exit(1)



In [13]:
from src.inference import (
    infer,
    process_with_logits,
    crop,
)
from src.detection import detect
import cv2 as cv
import os
from pathlib import Path

pathtoimg = Path("/content/dataset/finalDataset")
clases = ["2DMask", "3DMask", "Print", "Real", "Replay"]


correct = 0
total = 0

faildVids= []
for clas in clases:
    myclass = pathtoimg / clas
    for val in ["train", "val"]:
        if val == "train" : 
            continue
        myVal = myclass/"val"

        
        for file in os.listdir(myVal):
            cap = cv.VideoCapture(str(myVal/file))
            print(f"{str(myVal/file)}")
            if not cap.isOpened():
                faildVids.append(str(myVal/file))
                numOfFaild += 1 
                print("failed to open")
                continue 
            while cap.isOpened():
                res, image = cap.read()
                if not res:
                    break
                image_rgb = cv.cvtColor(image, cv.COLOR_BGR2RGB)

                faces = detect(image_rgb, face_detector, margin=5)
                if not faces:
                    continue 

                for face in faces:
                    bbox = face["bbox"]
                    x, y, w, h = bbox["x"], bbox["y"], bbox["width"], bbox["height"]
                    face_crop = crop(image_rgb, (x, y, x + w, y + h), 1.5)

                    pred = infer([face_crop], liveness_session, input_name, 128)[0]
                    result = process_with_logits(pred, logit_threshold)

                    is_correct = result["is_real"] == (clas == "Real")
                    correct += is_correct
                    total += 1

print(f"Accuracy: {correct/total:.4f}")

/content/dataset/finalDataset/2DMask/val/000069.mp4
/content/dataset/finalDataset/2DMask/val/000059.mp4
/content/dataset/finalDataset/2DMask/val/000058.mp4
/content/dataset/finalDataset/2DMask/val/000063.mp4
/content/dataset/finalDataset/2DMask/val/000064.mp4
/content/dataset/finalDataset/2DMask/val/000060.mp4
/content/dataset/finalDataset/2DMask/val/000067.mp4
/content/dataset/finalDataset/2DMask/val/000071.mp4
/content/dataset/finalDataset/2DMask/val/000072.mp4
/content/dataset/finalDataset/2DMask/val/000061.mp4
/content/dataset/finalDataset/2DMask/val/000068.mp4
/content/dataset/finalDataset/2DMask/val/000066.mp4
/content/dataset/finalDataset/2DMask/val/000065.mp4
/content/dataset/finalDataset/2DMask/val/000070.mp4
/content/dataset/finalDataset/2DMask/val/000062.mp4
/content/dataset/finalDataset/3DMask/val/000044.mp4
/content/dataset/finalDataset/3DMask/val/000051.mp4
/content/dataset/finalDataset/3DMask/val/000053.mp4
/content/dataset/finalDataset/3DMask/val/000047.mp4
/content/dat